In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
import os

In [4]:
# 使用小米 MiMo 模型
llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
    api_key=os.environ.get("MIMO_API_KEY"),
    temperature=0
)

In [5]:
# --- 定义工具 ---
@tool
def search_information(query: str) -> str:
    """搜索信息的工具，用于查找问题的答案，如天气、首都等。"""
    print(f"\n--- 工具调用: search_information，查询: '{query}' ---")
    # 模拟搜索工具的结果
    simulated_results = {
        "伦敦天气": "伦敦目前天气多云，温度约15°C。",
        "法国首都": "法国的首都是巴黎。",
        "地球人口": "地球人口约80亿。",
        "最高山峰": "珠穆朗玛峰是世界最高峰。",
    }
    result = simulated_results.get(query.lower(), f"模拟搜索结果：'{query}' - 未找到具体信息。")
    print(f"--- 工具结果: {result} ---")
    return result

tools = [search_information]


In [6]:
# --- 创建代理 ---
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="你是一个有用的助手。可以使用工具来查找信息。",
)


In [7]:
# --- 运行示例 ---

print("--- 示例1: 查询天气 ---")
inputs = {"messages": [{"role": "user", "content": "伦敦天气怎么样？"}]}
for chunk in agent.stream(inputs, stream_mode="updates"):
    print(chunk)

print("\n--- 示例2: 查询首都 ---")
inputs = {"messages": [{"role": "user", "content": "法国的首都是什么？"}]}
for chunk in agent.stream(inputs, stream_mode="updates"):
    print(chunk)

print("\n--- 示例3: 不需要工具的问题 ---")
inputs = {"messages": [{"role": "user", "content": "你好，今天过得怎么样？"}]}
for chunk in agent.stream(inputs, stream_mode="updates"):
    print(chunk)


--- 示例1: 查询天气 ---
{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 80, 'prompt_tokens': 270, 'total_tokens': 350, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 57, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None}}, 'model_provider': 'openai', 'model_name': 'mimo-v2.5-pro', 'system_fingerprint': None, 'id': 'ecce6cdce4fe4cf98ab24519621f7ed7', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019df33c-8e9c-7733-8777-6fbad5286aba-0', tool_calls=[{'name': 'search_information', 'args': {'query': '伦敦天气'}, 'id': 'call_fdeb7540498e49649bc84ead', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 270, 'output_tokens': 80, 'total_tokens': 350, 'input_token_details': {}, 'output_token_details': {'reasoning': 57}})]}}

--- 工具调用: search_information，查询: 

In [8]:
# --- 扩展：多个工具 ---

# 注意：以下工具都是模拟的（写死数据），实际应用中可调用真实API

@tool
def search_weather(city: str) -> str:
    """查询指定城市的天气信息。参数city是城市名称，如'北京'、'伦敦'。"""
    print(f"\n--- 工具调用: search_weather，城市: '{city}' ---")
    weather_data = {
        "北京": "北京今天晴天，温度25°C，空气质量良好。",
        "上海": "上海今天多云，温度28°C，湿度较高。",
        "伦敦": "伦敦今天阴天，温度15°C，可能有小雨。",
        "纽约": "纽约今天晴朗，温度22°C，微风。",
    }
    result = weather_data.get(city, f"未找到{city}的天气信息。")
    print(f"--- 工具结果: {result} ---")
    return result

@tool
def search_capital(country: str) -> str:
    """查询指定国家的首都。参数country是国家名称，如'中国'、'法国'。"""
    print(f"\n--- 工具调用: search_capital，国家: '{country}' ---")
    capital_data = {
        "中国": "中国的首都是北京。",
        "法国": "法国的首都是巴黎。",
        "日本": "日本的首都是东京。",
        "美国": "美国的首都是华盛顿。",
    }
    result = capital_data.get(country, f"未找到{country}的首都信息。")
    print(f"--- 工具结果: {result} ---")
    return result

@tool
def calculate(expression: str) -> str:
    """计算数学表达式。参数expression是数学表达式，如'2+3'、'10*5'。"""
    print(f"\n--- 工具调用: calculate，表达式: '{expression}' ---")
    try:
        result = str(eval(expression))
    except Exception as e:
        result = f"计算错误: {e}"
    print(f"--- 工具结果: {result} ---")
    return result

# 注册多个工具
multi_tools = [search_weather, search_capital, calculate]
print(f"已注册 {len(multi_tools)} 个工具: {[t.name for t in multi_tools]}")


已注册 3 个工具: ['search_weather', 'search_capital', 'calculate']


In [9]:
# --- 使用多个工具创建代理 ---

multi_agent = create_agent(
    model=llm,
    tools=multi_tools,
    system_prompt="你是一个有用的助手。可以使用工具查询天气、首都和进行计算。",
)

# 测试多个工具
print("--- 测试1: 查询天气 ---")
inputs = {"messages": [{"role": "user", "content": "北京天气怎么样？"}]}
for chunk in multi_agent.stream(inputs, stream_mode="updates"):
    print(chunk)

print("\n--- 测试2: 查询首都 ---")
inputs = {"messages": [{"role": "user", "content": "日本的首都是什么？"}]}
for chunk in multi_agent.stream(inputs, stream_mode="updates"):
    print(chunk)

print("\n--- 测试3: 计算 ---")
inputs = {"messages": [{"role": "user", "content": "计算 15 * 23 + 7"}]}
for chunk in multi_agent.stream(inputs, stream_mode="updates"):
    print(chunk)


--- 测试1: 查询天气 ---
{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 448, 'total_tokens': 490, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 19, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None}}, 'model_provider': 'openai', 'model_name': 'mimo-v2.5-pro', 'system_fingerprint': None, 'id': '046447dbc923411ca0a45b40a72dc392', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019df347-b855-72a0-86d5-2822e4d1ef4b-0', tool_calls=[{'name': 'search_weather', 'args': {'city': '北京'}, 'id': 'call_14398899a76449fca160c292', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 448, 'output_tokens': 42, 'total_tokens': 490, 'input_token_details': {}, 'output_token_details': {'reasoning': 19}})]}}

--- 工具调用: search_weather，城市: '北京' ---
--

In [ ]:
# --- 说明：模拟工具 vs 真实工具 ---
#
# 当前工具都是模拟的（写死数据）：
# - search_weather: 返回固定的天气数据
# - search_capital: 返回固定的首都数据
# - calculate: 使用eval()计算表达式
#
# 实际应用中，可以调用真实API：
# - search_weather: 调用天气API（如OpenWeatherMap）
# - search_capital: 调用地理数据库API
# - calculate: 使用安全的计算库
#
# 示例：真实天气API
# import requests
# @tool
# def search_weather(city: str) -> str:
#     """查询指定城市的天气信息"""
#     api_key = os.environ.get("WEATHER_API_KEY")
#     url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}"
#     response = requests.get(url)
#     data = response.json()
#     return f"{city}天气: {data['weather'][0]['description']}, 温度: {data['main']['temp']}K"
